# 1. feladat
Importáljuk a megfelelő modulokat az adatelemzéshez és vizualizációhoz. Az adatokat töltsük be a data.csv fájlból egy dataframe-be és ellenőrizzük a művelet eredményét. 2 pont

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data.csv')

df.head()

# 2. feladat
Adatok áttekintése: Vizsgáljuk meg az adatkészletet, hogy megértsük annak szerkezetét és az adattípusokat. Jelenítsük meg ezt az információt a dataframe-ről. 0,5 pont

In [ ]:
df.info()

# 3. feladat
Adatok előfeldolgozása: Végezzünk adatelőkészítést az elemzés előtt. Alakítsuk át a mintavételi dátumot ('Sampling Date') dátum-idő formátumra, és ellenőrizzük, hogy vannak-e hiányozó értékek a dataframe-ben. 1,5 pont

In [ ]:
df['Sampling Date'] = pd.to_datetime(df['Sampling Date'], errors='coerce')

df.isnull().sum()


# 4. feladat: 
Feltáró adatelemzés: Nézzük meg az adatokat néhány vizualizációval, hogy megértsük a szennyezőanyagok szintjeinek eloszlását és a szennyvízrendszerek állapotát a különböző földrajzi helyeken.

# 4.a
Ábrázoljuk a nitrogén és foszfor szintek eloszlását Matplotlib histogram-mal két subplot elhelyezésével egy közös ábrán. Az ábra méretét célszerű 12x6-ra beállítani.
A diagramokat lássuk el jelmagyarázattal, diagramcímmel és tengelyfeliratokkal. A diagramokban a KDE is kerüljön ábrázolásra. 2,5 pont

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(df['Nitrogen (mg/L)'], kde=True, color='blue', bins=10)
plt.title('Distribution of Nitrogen Levels')
plt.xlabel('Nitrogen (mg/L)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.histplot(df['Phosphorus (mg/L)'], kde=True, color='green', bins=10)
plt.title('Distribution of Phosphorus Levels')
plt.xlabel('Phosphorus (mg/L)')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

# 4.b
Ábrázoljuk a nitrogén szintek geográfiai eloszlását scatter-en, ahol a vízszintes tengelyen a Longitude és a függőleges tengelyen a Latitude értékek szerepeljenek. A hue paramétert a 'Nitrogen (mg/L)' értékekre állítsuk be, és használjunk 'coolwarm' színpalettát. 2 pont

In [ ]:
plt.figure(figsize=(8, 4))

sns.scatterplot(
    data=df,
    x='Geographical Location (Longitude)',
    y='Geographical Location (Latitude)',
    hue='Nitrogen (mg/L)',
    palette='coolwarm',
)
plt.title('Geographical Distribution of Nitrogen Levels')

plt.show()

# 4.c
Ábrázoljuk a foszfor szintek geográfiai eloszlását scatter-en, ahol a vízszintes tengelyen a Longitude és a függőleges tengelyen a Latitude értékek szerepeljenek. A hue paramétert a 'Phosphorus (mg/L)' értékekre állítsuk be, és használjunk itt is 'coolwarm' színpalettát. 1 pont

In [ ]:
plt.figure(figsize=(8, 4))

sns.scatterplot(
    data=df,
    x='Geographical Location (Longitude)',
    y='Geographical Location (Latitude)',
    hue='Phosphorus (mg/L)',
    palette='coolwarm',
)
plt.title('Geographical Distribution of Phosphorous Levels')

plt.show()

# 5. feladat
Ábrázoljuk a nitrogén szinteket valós térképen: Hozzuk létre a map objektumot és jelenítsük meg az interaktív térképet az adott GPS koordinátákkal együtt. 3 pont

In [ ]:
from matplotlib import colors
import folium

max_N = round(df['Nitrogen (mg/L)'].max())
print(max_N)
levels = range(max_N+1)
color_dict = dict(zip(levels, list(colors.cnames.values())[0:-1:10]))
color_dict

map_center = [df['Geographical Location (Latitude)'].mean(), df['Geographical Location (Longitude)'].mean()]
m = folium.Map(location=map_center, zoom_start=4, tiles='OpenStreetMap')

for ind, row in df.iterrows():
    N_level = int(round(row['Nitrogen (mg/L)']))
    folium.CircleMarker(
        location=[row['Geographical Location (Latitude)'], row['Geographical Location (Longitude)']],
        color=color_dict.get(N_level), 
        fill=True,
    ).add_to(m)

m


# 6. feladat
A szennyvízrendszer állapota: Készítsünk gyakoriság diagramot a State of Sewage System lehetséges állapotairól. Használjuk a Seaborn histplot metódusát és a kép méretét 6x6-osra állítsuk be. A diagramot lássuk el jelmagyarázattal, diagramcímmel és tengelyfeliratokkal. 2,5 pont

In [ ]:
plt.figure(figsize=(6,6))

sns.histplot(
    data=df,
    x='State of Sewage System',
    color='blue'
)

plt.title('A szennyvízrendszer állapota')
plt.xlabel('State of Sewage System')
plt.ylabel('Count')

plt.show()


# 7. feladat: Korrelációs elemzés

## 7.a
Vizsgáljuk meg az adatkészlet numerikus változói közötti korrelációt, hogy azonosítani tudjuk az esetleges kapcsolatokat. Először hozzuk létre az N+P oszlopot, ahol a teljes mért szennyezés értékét (a nitrogén és foszfor mennyiségek összegét) jelenítjük meg a mérési helyeken, a mért időpontokban. 1 pont

In [ ]:
df['N+P'] = df['Nitrogen (mg/L)'] + df['Phosphorus (mg/L)']
df[['Geographical Location (Latitude)', 'Geographical Location (Longitude)', 'Sampling Date', 'N+P']].head()

## 7.b
A szennyvízrendszer állapotát megadó State of Sewage System oszlop adatait konvertáljuk számértértékké a LabelEncoder alkalmazásával. Ellenőrizzük az eredményt. 2 pont

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['State of Sewage System'] = le.fit_transform(df['State of Sewage System'])

df['State of Sewage System'].head()

## 7.c
Válogassuk ki a csak numerikus adatokat tartalmazó oszlopokat egy új dataframe-be, végezzük el a korreláció vizsgálatát a pearson módszer alkalmazásával, és jelenítsük meg a korrelációs mátrixot. 3 pont

In [ ]:
numerical_df = df.select_dtypes(include='number')

corr_matrix = numerical_df.corr(method='pearson')

corr_matrix

## 7.d
A korreláció vizualizációhoz használjunk heatmap diagramot, a korrelációs együtthatók megjelenítésével (2 tizedesjegyre kerekítve), coolwarm színtérképpel, 8x6-os képmérettel és diagramcímmel. 2,5 pont

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
)

plt.title('Correlation Heatmap')
plt.show()

# 8. feladat
Rövidítsük a mezőneveket a következőképpen:
Lati = Geographical Location (Latitude)
Long = Geographical Location (Longitude)
SDate = Sampling Date
N = Nitrogen (mg/L)
P = Phosphorus (mg/L)
SWS = State of Sewage System
Ellenőrizzük az info() metódussal, hogy sikeres volt-e az átnevezés. 1,5 pont

In [ ]:
df = df.rename(columns={
    'Geographical Location (Latitude)': 'Lati',
    'Geographical Location (Longitude)': 'Long',
    'Sampling Date': 'SDate',
    'Nitrogen (mg/L)': 'N',
    'Phosphorus (mg/L)': 'P',
    'State of Sewage System': 'SWS'
})

df.info()

# 9. feladat
Készítsük el az adatok statisztikai jellemzését: Használjuk a describe() utasítást. 1 pont

In [ ]:
df.describe()

# 10. feladat
Vizsgáljuk meg azokat a helyeket, ahol a N+P koncentráció magas. Ehhez készítsünk egy Q (quality) oszlopot az apply függvény alkalmazásával, a következő értékekkel: 1, ha N+P 4-nél kisebb; 3, ha 10-nél nagyobb; a közbeeső tartományra az érték legyen 2. Ellenőrizzük az eredményt. 2,5 pont

In [ ]:
def q(np):
    if np < 4:
        return 1
    elif np > 10:
        return 3
    else:
        return 2

df['Q'] = df['N+P'].apply(q)
df[['N+P', 'Q']].head(15)

# 11. feladat
Csoportosítsuk a Q oszlop értékeit és írassuk ki a csoportok méretét. 1,5 pont

In [ ]:
df.groupby('Q').size()

# 12. feladat
Jelenítsük meg a Q értékek geográfiai eloszlását Seaborn scatterplot-tal, coolwarm színpalettával. A kép méretét 10x6-osra és a hue paramétert a Q értékekre állítsuk be. A diagramot lássuk el jelmagyarázattal, diagramcímmel és tengelyfeliratokkal. 1 pont

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='Long',
    y='Lati',
    hue='Q',
    palette='coolwarm',
)
plt.title('Geographical Distribution of Q Levels')

plt.show()

# 13. feladat
A következő részben készítsünk egy elemzést a DecisionTree modellel, a kapott modell értékelésével, és a döntési fa megjelenítésével. Importáljuk a szükséges modulokat és osztályokat a Scikit-learn könyvtárból.
Ebben az elemzésben nem használjuk a dátum adatokat, ezért a vizsgálathoz hozzunk létre egy új dataframe-et az SDate oszlop nélkül. Ellenőrizzük az új dataframe-et.
A modell alkalmazásához erre az új dataframe-re válasszuk szét az adathalmazt független (`x`) és függő (`y`) változókra, az y értékeknek az SWS oszlopot választva. 4,5 pont

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df2 = df.drop(columns=['SDate'])
df2.info()

x=df2.drop(columns=['SWS'])
y=df2['SWS']

# 14. feladat
Adatok felosztása, modellépítés és betanítás. Vizualizáció
Osszuk fel az adatokat 4:1 arányban véletlenszerűen train és test adatokra, és tanítsuk be a modellünket. Vizualizáljuk a döntési fát a Matplotlib segítségével, az ábra méretét célszerű 12x8-ra beállítani. 4 pont

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, test_size=0.2)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(x_train, y_train)

plt.figure(figsize=(12, 8))
plot_tree(model, filled=True, feature_names=x.columns, class_names=['Poor', 'Moderate', 'Good'])
plt.show()

# 15.feladat: A modell értékelése
Készítsünk predikciót a teszt halmazon. Számítsuk ki a pontosságot. Generáljunk osztályozási riportot és konfúziós mátrixot és ezeket jelenítsük meg. 4 pont

In [ ]:

y_pred = model.predict(x_test)

accuracy_score(y_test, y_pred)

display(classification_report(y_test, y_pred))
display(confusion_matrix(y_test, y_pred))

# 16. feladat
A népesség és a szennyezettség mértékének kapcsolatának vizsgálata Lineáris Regresszióval
A következő részben készítsünk egy elemzést a LinearRegression modellel, a kapott modell értékelésével, és a scatter diagram megjelenítésével. Először importáljuk a szükséges modulokat és osztályokat a Scikit-learn könyvtárból. Független változónak a Population oszlopot, és függő változónak az N+P oszlopot válasszuk.
Osszuk fel az adatokat 4:1 arányban véletlenszerűen train és test adatokra, és tanítsuk be a modellünket. Vizualizáljuk a teszt adatokat és a lineáris regressziót is jelenítésük meg a Matplotlib scatter alkalmazásával. 8,5 pont

In [ ]:
from sklearn.linear_model import LinearRegression

x = df[['Population']]
y = df['N+P']

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, test_size=0.2)

lr = LinearRegression()
lr.fit(x_train, y_train)

plt.figure(figsize=(8, 6))
plt.scatter(x, y, color='red')
plt.plot(x, lr.predict(x), color='blue')
plt.xlabel('Population (teszt halmaz)')
plt.ylabel('N+P (teszt halmaz)')
plt.title('Linear Regression a teszt adatokra')
plt.show()

# 17. feladat
A modell értékelése
Készítsünk predikciót a teszt halmazon, és jelenítsük meg az első öt predikció, és az első öt valós y teszt értékeket. Írassuk ki a betanított modellünk pontosságát is. 4 pont

In [ ]:
y_pred = lr.predict(x_test)

display(y_pred[:5])
display(y_test.values[:5])

display(lr.score(x_test, y_test))